# DESI Data Growth

## Introduction

This notebook uses intermediate metadata files produced by JSON to measure data growth. The primary purpose is to detect large changes in data use over relatively short time periods, *e.g.* a couple of weeks.

The intermediate JSON files are produced by `convert_metadata.sh` in the `decamUtil` package. `convert_metadata.sh` is run daily as a cron job by the `desi` collaboration account on `dtn02.nersc.gov`.

## Imports

Note that the decamUtil package currently must be added "by hand" since it is not part of the standard DESI software distribution.

In [ ]:
%matplotlib inline
import os
import sys
sys.path.insert(0, os.path.join(os.environ['HOME'], 'Documents', 'Code', 'svn', 'DESI', 'decam', 'code', 'decamUtil', 'trunk', 'py'))
import re
import datetime
import json
import gzip
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from decamUtil.metadata import Directory, DirectoryDecoder

## Configuration

Specify start date, end date, cadence and output directory.

In [ ]:
start_date = datetime.date(2026, 1, 31)
end_date = datetime.date(2026, 2, 17)
cadence = 2  # Every N days
figure_dir = os.path.join(os.environ['DESI_ROOT'], 'users', os.environ['USER'], 'data_growth')

## Define plot helper function

In [ ]:
def plot_growth(data, filesystem, directory, end_date, start_index=0, xlim=None, log=False, model=None):
    """Creates a cumulative plot of data volume and number of files (inodes) for a particular data set.
    
    Parameters
    ----------
    data : :class:`dict`
        The full growth data set.
    filesystem : :class:`str`
        The filesystem that contains the data.
    directory : :class:`str`
        The directory path within `filesystem` to plot.
    end_date : :class:`str`
        The effective end date of the growth data set.
    start_index : :class:`int`, optional
        If set, start plotting the growth data with this index.
    xlim : :class:`tuple`, optional
        If set, use these *two* dates for the x range of the plot.
    log : :class:`boolean`, optional
        If ``True`` use a logarithmic Y-axis.
    model : :class:`tuple`, optional
        A tuple of x, y, label data to overplot.

    Returns
    -------
    :class:`tuple`
        A tuple of figure and axis handles.
    """
    if not directory.startswith(filesystem+'/'):
        directory = filesystem + '/' + directory
    growth_dates = np.array(sorted(data.keys()), dtype=np.datetime64)
    growth_files = np.array([data[d][filesystem][directory].files for d in sorted(data.keys())])
    growth_bytes = np.array([data[d][filesystem][directory].size for d in sorted(data.keys())])
    # growth_dates = np.array(sorted(data[directory].growth.keys()), dtype=np.datetime64)
    # growth_files = np.cumsum(np.array([data[directory].growth[k][0] for k in sorted(data[directory].growth.keys())], dtype=np.int64))
    # growth_bytes = np.cumsum(np.array([data[directory].growth[k][1] for k in sorted(data[directory].growth.keys())], dtype=np.int64))
    GB = 1024*1024*1024
    fig, ax1 = plt.subplots(nrows=1, ncols=1, figsize=(16,9), dpi=100)
    if log:
        p1 = ax1.semilogy(growth_dates[start_index:], growth_bytes[start_index:]/GB, color='black', label='Data Volume')
    else:
        p1 = ax1.plot(growth_dates[start_index:], growth_bytes[start_index:]/GB, color='black', label='Data Volume')
    ax2 = ax1.twinx()
    if log:
        p2 = ax2.semilogy(growth_dates[start_index:], growth_files[start_index:], color='blue', label='Number of Files')
    else:
        p2 = ax2.plot(growth_dates[start_index:], growth_files[start_index:], color='blue', label='Number of Files')
    if model is not None:
        p3 = ax1.plot(model[0], model[1], color='black', linestyle='--', label=model[2])
    foo = ax1.set_xlabel('Date')
    foo = ax1.set_ylabel('Cumulative Data [GB]')
    foo = ax1.set_title(f'{directory} as of {end_date}')
    foo = ax2.set_ylabel('Cumulative Number of Files', color='blue')
    foo = ax2.tick_params(axis='y', labelcolor='blue')
    # Major ticks every 6 months.
    # fmt_half_year = mdates.MonthLocator(interval=3)
    # foo = ax1.xaxis.set_major_locator(fmt_half_year)
    # Major ticks every week.
    fmt_week = mdates.WeekdayLocator(byweekday=1)
    foo = ax1.xaxis.set_major_locator(fmt_week)
    # Minor ticks every month.
    # fmt_month = mdates.MonthLocator()
    # foo = ax1.xaxis.set_minor_locator(fmt_month)
    # Minor ticks every day.
    # fmt_day = mdates.DayLocator()
    # foo = ax1.xaxis.set_major_locator(fmt_day)
    # Format the coords message box, i.e. the numbers displayed as the cursor moves
    # across the axes within the interactive GUI.
    # Text in the x axis will be displayed in 'YYYY-mm' format.
    foo = ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    foo = ax1.format_xdata = mdates.DateFormatter('%Y-%m-%d')
    foo = ax1.grid(True)
    foo = ax2.yaxis.grid(linestyle=':')
    # Rotates and right aligns the x labels, and moves the bottom of the
    # axes up to make room for them.
    foo = fig.autofmt_xdate()
    if xlim is not None:
        foo = ax1.set_xlim(np.datetime64(xlim[0]), np.datetime64(xlim[1]))
        foo = ax2.set_xlim(np.datetime64(xlim[0]), np.datetime64(xlim[1]))
    # Data Volume legend in upper left
    foo = ax1.legend(loc=2, numpoints=1)
    # Number of files legend in lower right
    foo = ax2.legend(loc=4, numpoints=1)
    return (fig, ax1, ax2)

## Load JSON files

In [ ]:
date = start_date
data = dict()
while date <= end_date:
    d = date.strftime('%Y-%m-%d')
    if os.path.isfile(os.path.join(os.environ['DESI_ROOT'], 'metadata', f'{d}.json.gz')):
        with gzip.open(os.path.join(os.environ['DESI_ROOT'], 'metadata', f'{d}.json.gz')) as j:
            data[d] = json.load(j, object_hook=DirectoryDecoder)
    elif os.path.isfile(os.path.join(os.environ['DESI_ROOT'], 'metadata', f'{d}.json')):
        with open(os.path.join(os.environ['DESI_ROOT'], 'metadata', f'{d}.json')) as j:
            data[d] = json.load(j, object_hook=DirectoryDecoder)
    else:
        print(f"No file matching {d}!")
    date += datetime.timedelta(days=cadence)

## Plot some standard directories

In [ ]:
f, a1, a2 = plot_growth(data, 'desi', 'spectro/data', end_date.strftime('%Y-%m-%d'))
# f.savefig(os.path.join(figure_dir, f"desi_spectro_data_{end_date.strftime('%Y%m%d')}.png"))

In [ ]:
f, a1, a2 = plot_growth(data, 'desi', 'desi/spectro/redux/daily', end_date.strftime('%Y-%m-%d'))
# f.savefig(os.path.join(figure_dir, f"desi_spectro_redux_daily_{end_date.strftime('%Y%m%d')}.png"))

In [ ]:
f, a1, a2 = plot_growth(data, 'desicollab', 'cosmosim', end_date.strftime('%Y-%m-%d'))
# f.savefig(os.path.join(figure_dir, f"desicollab_cosmosim_{end_date.strftime('%Y%m%d')}.png"))

In [ ]:
f, a1, a2 = plot_growth(data, 'desicollab', 'mocks', end_date.strftime('%Y-%m-%d'))
# f.savefig(os.path.join(figure_dir, f"desicollab_mocks_{end_date.strftime('%Y%m%d')}.png"))

In [ ]:
f, a1, a2 = plot_growth(data, 'desicollab', 'science', end_date.strftime('%Y-%m-%d'))
# f.savefig(os.path.join(figure_dir, f"desicollab_science_{end_date.strftime('%Y%m%d')}.png"))

In [ ]:
f, a1, a2 = plot_growth(data, 'desicollab', 'users', end_date.strftime('%Y-%m-%d'))
# f.savefig(os.path.join(figure_dir, f"desicollab_users_{end_date.strftime('%Y%m%d')}.png"))

## Sample file change calculation

Because of occasional missing data, the dates with data do not necessarily correspond exactly to `start_date` and `end_date` above.

In [ ]:
delta_filesystem = 'desicollab'
delta_directory = 'desicollab/mocks'
actual_dates = list(sorted(data.keys()))
delta_files = data[actual_dates[-1]][delta_filesystem][delta_directory].files - data[actual_dates[0]][delta_filesystem][delta_directory].files
delta_bytes = data[actual_dates[-1]][delta_filesystem][delta_directory].size - data[actual_dates[0]][delta_filesystem][delta_directory].size
print(delta_directory)
print(f"{actual_dates[0]} to {actual_dates[-1]}")
print(f"{delta_files} inodes added")
print(f"{delta_bytes/1024.0/1024.0/1024.0} GB added")

## Find individual users with large changes

In [ ]:
top_users = re.compile(r'desicollab/users/[^/]+$')
# Assume that directories are very rarely deleted, so all users will be present at the end.
all_users = set([d for d in data[actual_dates[-1]]['desicollab'].keys() if top_users.match(d) is not None])
common_users = set([d for d in data[actual_dates[0]]['desicollab'].keys() if top_users.match(d) is not None]) & all_users

In [ ]:
user_delta_files = sorted([(u.split('/')[-1], data[actual_dates[-1]]['desicollab'][u].files - data[actual_dates[0]]['desicollab'][u].files) for u in common_users], key=lambda x: x[1], reverse=True)
user_delta_files_onek = [u for u in user_delta_files if u[1] > 1000]
user_delta_files_onek

In [ ]:
user_delta_bytes = sorted([(u.split('/')[-1], data[actual_dates[-1]]['desicollab'][u].size - data[actual_dates[0]]['desicollab'][u].size) for u in common_users], key=lambda x: x[1], reverse=True)
user_delta_bytes_gb = [(u[0], u[1]/1024.0/1024.0/1024.0) for u in user_delta_bytes if u[1] > 1000000000]
user_delta_bytes_gb

In [ ]:
user_total_files = sorted([(u.split('/')[-1], data[actual_dates[-1]]['desicollab'][u].files) for u in all_users], key=lambda x: x[1], reverse=True)
user_total_files_100k = [u for u in user_total_files if u[1] > 100000]
user_total_files_100k

In [ ]:
user_total_bytes = sorted([(u.split('/')[-1], data[actual_dates[-1]]['desicollab'][u].size) for u in all_users], key=lambda x: x[1], reverse=True)
user_total_bytes_gb = [(u[0], u[1]/1024.0/1024.0/1024.0) for u in user_total_bytes if u[1] > 1000000000]
user_total_bytes_gb

In [ ]:
username = 'desi'
f, a1, a2 = plot_growth(data, 'desicollab', f'users/{username}', end_date.strftime('%Y-%m-%d'))
# f.savefig(os.path.join(figure_dir, f"desicollab_users_{username}_{end_date.strftime('%Y%m%d')}.png"))